# AA test tutorial 
AA test is important part of randomized controlled experiment, for example AB test. 

The objectives of the AA test are to verify the assumption of uniformity of samples as a result of the applied partitioning method, to select the best partition from the available ones, and to verify the applicability of statistical criteria for checking uniformity. 

For example, there is a hypothesis about the absence of dependence of features on each other. If this hypothesis is not followed, the AA test will fail.

[Wiki AA test](https://github.com/sb-ai-lab/HypEx/wiki/%D0%90%D0%90-Test) with more detailed description of terms for AA test.

<ul>
  <li><a href="#creation-of-a-new-test-dataset-with-synthetic-data">Creation of a new test dataset with synthetic data.
  <li><a href="#one-split-of-aa-test">One split of AA test.
  <li><a href="#aa-test">AA test.
  <li><a href="#aa-test-with-stratification">AA test with stratification.
</ul>

In [1]:
from hypex import AATest
from hypex.dataset import (
    ConstGroupRole,
    Dataset,
    InfoRole,
    StratificationRole,
    TargetRole,
    TreatmentRole,
)
from hypex.utils import create_test_data

/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/__init__.py:50: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


## Creation of a new test dataset with synthetic data. 

In order to be able to work with our data in HypEx, first we need to convert it into `dataset`. It is important to mark the data fields by assigning the appropriate `roles`:
- TargetRole: a role for columns that contain features or predictor variables. Our split will be based on them. Applied by default if the role is not specified for the column.
- TreatmentRole: a role for columns that show the treatment or intervention.
- InfoRole: a role for columns that contain information about the data, such as user IDs. 

In [2]:
data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": StratificationRole(str),
    }, data=create_test_data(),
)
data

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry
0,0.0,0.0,0.0,487.5,413.0,47.0,F,Logistics
1,1.0,9.0,1.0,502.0,442.888889,42.0,F,Logistics
2,2.0,0.0,0.0,490.0,419.222222,42.0,M,Logistics
3,3.0,0.0,0.0,489.0,423.0,61.0,F,E-commerce
4,4.0,1.0,1.0,548.0,520.444444,65.0,M,E-commerce
...,...,...,...,...,...,...,...,...
9995,9995.0,3.0,1.0,488.5,514.888889,21.0,F,Logistics
9996,9996.0,0.0,0.0,473.5,413.888889,51.0,M,Logistics
9997,9997.0,0.0,0.0,486.5,413.666667,37.0,M,Logistics
9998,9998.0,8.0,1.0,501.5,462.222222,20.0,F,Logistics


In [3]:
data.roles

{'user_id': Info(<class 'int'>),
 'pre_spends': Target(<class 'float'>),
 'post_spends': Target(<class 'float'>),
 'gender': Stratification(<class 'str'>),
 'signup_month': Default(<class 'float'>),
 'treat': Default(<class 'float'>),
 'age': Default(<class 'float'>),
 'industry': Default(<class 'object'>)}

## AA test
Then we run the experiment on our prepared dataset, wrapped into ExperimentData. In this case we select one of the pre-assembled pipeline, AA_TEST.
We can set the number of iterations for simple execution. In this case the random states are the numbers of each iteration.

In [4]:
test = AATest(n_iterations=10)
result = test.execute(data)

100%|██████████| 10/10 [00:01<00:00,  7.31it/s]
[DEBUG _compute_weighted_pvalues] pval_cols = ['pre_spends TTest p-value test_1', 'post_spends TTest p-value test_1', 'pre_spends KSTest p-value test_1', 'post_spends KSTest p-value test_1', 'mean TTest p-value all', 'mean KSTest p-value all']
[DEBUG _compute_weighted_pvalues] col='pre_spends TTest p-value test_1', lookup_key='pre_spends TTest test_1', weight=0.95, NaN=0/10, values=[0.8649779651053894, 0.9916075572373465, 0.19406034893642896, 0.5545227786361993, 0.7142073907415776, 0.9143111230148722, 0.7900405707533317, 0.6679413416413083, 0.8914543231508143, 0.2712485566696564]
[DEBUG _compute_weighted_pvalues] col='post_spends TTest p-value test_1', lookup_key='post_spends TTest test_1', weight=0.95, NaN=0/10, values=[0.708159724739563, 0.9126422564634091, 0.9383324312278087, 0.9554425139993918, 0.23142812365394647, 0.7129285917476822, 0.9354627424514447, 0.08190475171031816, 0.514290604109518, 0.7545020060691605]
[DEBUG _compute_weigh

In [5]:
result.resume

,feature,group,TTest aa test,KSTest aa test,TTest best split,KSTest best split,result,control mean,test mean,difference,difference %
0,post_spends,test_1,OK,OK,OK,OK,OK,452.173332,452.081878,-0.091455,-0.020226
1,pre_spends,test_1,OK,OK,OK,OK,OK,487.410630,487.406429,-0.004201,-0.000862


**Interpretation of AA test results**

Each row in the table corresponds to a target feature being tested for equality between the control and test groups. Two statistical tests are used:

- **TTest**: tests if means are statistically different.
- **KSTest**: tests if distributions differ.

The `OK` / `NOT OK` labels show whether the difference is statistically significant. A `NOT OK` result indicates a possible imbalance.

Typical threshold:
- If p-value < 0.05 → `NOT OK` (statistically significant difference)
- If p-value ≥ 0.05 → `OK` (no significant difference)

If any metric has a `NOT OK` status in the `AA test` column, it means at least one iteration showed significant difference.


In [6]:
result.aa_score

,score,pass
pre_spends TTest test_1,0.95,True
post_spends TTest test_1,0.95,True
pre_spends KSTest test_1,0.95,True
post_spends KSTest test_1,0.95,True


**Interpreting `aa_score`**

This output shows p-values and the overall pass/fail status for each test type and feature. A high p-value (close to 1.0) means the test passed — the groups are similar.

- `score`: p-value of the statistical test.
- `pass`: True if no iterations showed significant differences.

Note: Even if the average p-value is high, the `pass` might still be False if at least one of the iterations had a p-value < 0.05.


In [7]:
result.best_split

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry,split
0,0,0.0,0.0,487.5,413.0,47.0,F,Logistics,control
1,1,9.0,1.0,502.0,442.888889,42.0,F,Logistics,test_1
2,2,0.0,0.0,490.0,419.222222,42.0,M,Logistics,test_1
3,3,0.0,0.0,489.0,423.0,61.0,F,E-commerce,control
4,4,1.0,1.0,548.0,520.444444,65.0,M,E-commerce,control
...,...,...,...,...,...,...,...,...,...
8996,9995,3.0,1.0,488.5,514.888889,21.0,F,Logistics,test_1
8997,9996,0.0,0.0,473.5,413.888889,51.0,M,Logistics,test_1
8998,9997,0.0,0.0,486.5,413.666667,37.0,M,Logistics,control
8999,9998,8.0,1.0,501.5,462.222222,20.0,F,Logistics,test_1


**About `best_split`**

This shows the best found split of the dataset, where control and test groups are as similar as possible in terms of target metrics.

You can use this split for future modeling or as a validation check before proceeding to actual experiments.


In [8]:
result.best_split_statistic

,feature,group,control mean,test mean,difference,difference %,TTest pass,TTest p-value,KSTest pass,KSTest p-value
0,post_spends,test_1,452.173332,452.081878,-0.091455,-0.020226,OK,0.912642,OK,0.985289
1,pre_spends,test_1,487.410630,487.406429,-0.004201,-0.000862,OK,0.991608,OK,0.680054


**Understanding `best_split_statistic`**

This table contains detailed statistics for the best (most balanced) split found across all iterations. You can compare:

- Mean values in control vs test group.
- Absolute and relative differences.
- p-values for both tests.

Ideally, all rows should have `OK` in both TTest and KSTest columns, and small difference values (<1%).

In [9]:
result.experiments

,splitter_id,pre_spends GroupDifference control mean test_1,pre_spends GroupDifference test mean test_1,pre_spends GroupDifference difference test_1,pre_spends GroupDifference difference % test_1,post_spends GroupDifference control mean test_1,post_spends GroupDifference test mean test_1,post_spends GroupDifference difference test_1,post_spends GroupDifference difference % test_1,pre_spends TTest p-value test_1,...,post_spends TTest pass test_1,pre_spends KSTest p-value test_1,pre_spends KSTest pass test_1,post_spends KSTest p-value test_1,post_spends KSTest pass test_1,mean TTest p-value,mean TTest pass,mean KSTest p-value,mean KSTest pass,mean test score
0,AASplitter┴rs 0┴,487.374918,487.442835,0.067917,0.013935,451.972839,452.284891,0.312052,0.069042,0.864978,...,False,0.917718,False,0.789246,False,0.786569,0.0,0.853482,0.0,0.831178
1,AASplitter┴rs 1┴,487.410630,487.406429,-0.004201,-0.000862,452.173332,452.081878,-0.091455,-0.020226,0.991608,...,False,0.680054,False,0.985289,False,0.952125,0.0,0.832672,0.0,0.872489
2,AASplitter┴rs 2┴,487.151902,487.670710,0.518808,0.106498,452.159083,452.094589,-0.064493,-0.014263,0.194060,...,False,0.417594,False,0.732330,False,0.566196,0.0,0.574962,0.0,0.572040
3,AASplitter┴rs 3┴,487.527417,487.291308,-0.236109,-0.048430,452.103725,452.150305,0.046580,0.010303,0.554523,...,False,0.898989,False,0.606782,False,0.754983,0.0,0.752885,0.0,0.753584
4,AASplitter┴rs 4┴,487.334935,487.481224,0.146289,0.030018,451.625490,452.623003,0.997512,0.220872,0.714207,...,False,0.715436,False,0.033912,True,0.472818,0.0,0.374674,0.5,0.407389
5,AASplitter┴rs 5┴,487.429898,487.386917,-0.042982,-0.008818,451.974569,452.281263,0.306694,0.067857,0.914311,...,False,0.896790,False,0.877596,False,0.813620,0.0,0.887193,0.0,0.862669
6,AASplitter┴rs 6┴,487.354936,487.461292,0.106356,0.021823,452.161182,452.093687,-0.067495,-0.014927,0.790041,...,False,0.463898,False,0.443389,False,0.862752,0.0,0.453643,0.0,0.590013
7,AASplitter┴rs 7┴,487.322771,487.494116,0.171346,0.035161,451.401601,452.851638,1.450037,0.321230,0.667941,...,False,0.956764,False,0.215402,False,0.374923,0.0,0.586083,0.0,0.515696
8,AASplitter┴rs 8┴,487.381228,487.435738,0.054510,0.011184,452.399268,451.855642,-0.543627,-0.120165,0.891454,...,False,0.403851,False,0.432425,False,0.702872,0.0,0.418138,0.0,0.513049
9,AASplitter┴rs 9┴,487.187080,487.626571,0.439491,0.090210,451.995845,452.256523,0.260678,0.057673,0.271249,...,False,0.130045,False,0.423201,False,0.512875,0.0,0.276623,0.0,0.355374


# AA Test with random states

We can also adjust some of the preset parameters of the experiment by assigning them to the respective params of the experiment. I.e. here we set the range of the random states we want to run our AA test for. 

In [10]:
data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": StratificationRole(str),
    }, data=create_test_data(),
)
data

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry
0,0.0,0.0,0.0,467.0,420.666667,18.0,F,E-commerce
1,1.0,6.0,1.0,517.5,479.111111,34.0,M,E-commerce
2,2.0,6.0,1.0,503.5,495.333333,62.0,F,E-commerce
3,3.0,0.0,0.0,484.5,422.222222,53.0,M,Logistics
4,4.0,0.0,0.0,484.0,415.222222,56.0,F,Logistics
...,...,...,...,...,...,...,...,...
9995,9995.0,6.0,1.0,486.5,485.444444,65.0,M,Logistics
9996,9996.0,0.0,0.0,470.5,419.111111,45.0,F,Logistics
9997,9997.0,0.0,0.0,488.5,419.666667,22.0,M,E-commerce
9998,9998.0,10.0,1.0,480.0,443.222222,52.0,F,Logistics


In [11]:
test = AATest(random_states=[56, 72, 2, 43])
result = test.execute(data)

100%|██████████| 4/4 [00:00<00:00,  7.17it/s]
[DEBUG _compute_weighted_pvalues] pval_cols = ['pre_spends TTest p-value test_1', 'post_spends TTest p-value test_1', 'pre_spends KSTest p-value test_1', 'post_spends KSTest p-value test_1', 'mean TTest p-value all', 'mean KSTest p-value all']
[DEBUG _compute_weighted_pvalues] col='pre_spends TTest p-value test_1', lookup_key='pre_spends TTest test_1', weight=0.95, NaN=0/4, values=[0.8974426165654917, 0.9019187500179193, 0.7699324449490181, 0.01935428219647836]
[DEBUG _compute_weighted_pvalues] col='post_spends TTest p-value test_1', lookup_key='post_spends TTest test_1', weight=0.95, NaN=0/4, values=[0.2578155832641702, 0.9263169121343993, 0.2891075987069418, 0.9483501056553223]
[DEBUG _compute_weighted_pvalues] col='pre_spends KSTest p-value test_1', lookup_key='pre_spends KSTest test_1', weight=0.95, NaN=0/4, values=[0.9783798188121037, 0.763879350971972, 0.9222558597072193, 0.14259934955569684]
[DEBUG _compute_weighted_pvalues] col='pos

In [12]:
result.resume

,feature,group,TTest aa test,KSTest aa test,TTest best split,KSTest best split,result,control mean,test mean,difference,difference %
0,post_spends,test_1,OK,OK,OK,OK,OK,451.997959,452.074520,0.076561,0.016938
1,pre_spends,test_1,OK,OK,OK,OK,OK,487.177694,487.227354,0.049660,0.010193


In [13]:
result.aa_score

,score,pass
pre_spends TTest test_1,0.95,True
post_spends TTest test_1,0.95,True
pre_spends KSTest test_1,0.95,True
post_spends KSTest test_1,0.95,True


In [14]:
result.best_split

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry,split
0,0,0.0,0.0,467.0,420.666667,18.0,F,E-commerce,control
1,1,6.0,1.0,517.5,479.111111,34.0,M,E-commerce,test_1
2,2,6.0,1.0,503.5,495.333333,62.0,F,E-commerce,test_1
3,3,0.0,0.0,484.5,422.222222,53.0,M,Logistics,test_1
4,4,0.0,0.0,484.0,415.222222,56.0,F,Logistics,control
...,...,...,...,...,...,...,...,...,...
8996,9995,6.0,1.0,486.5,485.444444,65.0,M,Logistics,test_1
8997,9996,0.0,0.0,470.5,419.111111,45.0,F,Logistics,test_1
8998,9997,0.0,0.0,488.5,419.666667,22.0,M,E-commerce,control
8999,9998,10.0,1.0,480.0,443.222222,52.0,F,Logistics,control


In [15]:
result.best_split_statistic

,feature,group,control mean,test mean,difference,difference %,TTest pass,TTest p-value,KSTest pass,KSTest p-value
0,post_spends,test_1,451.997959,452.074520,0.076561,0.016938,OK,0.926317,OK,0.470399
1,pre_spends,test_1,487.177694,487.227354,0.049660,0.010193,OK,0.901919,OK,0.763879


In [16]:
result.experiments

,splitter_id,pre_spends GroupDifference control mean test_1,pre_spends GroupDifference test mean test_1,pre_spends GroupDifference difference test_1,pre_spends GroupDifference difference % test_1,post_spends GroupDifference control mean test_1,post_spends GroupDifference test mean test_1,post_spends GroupDifference difference test_1,post_spends GroupDifference difference % test_1,pre_spends TTest p-value test_1,...,post_spends TTest pass test_1,pre_spends KSTest p-value test_1,pre_spends KSTest pass test_1,post_spends KSTest p-value test_1,post_spends KSTest pass test_1,mean TTest p-value,mean TTest pass,mean KSTest p-value,mean KSTest pass,mean test score
0,AASplitter┴rs 56┴,487.228977,487.177027,-0.051951,-0.010662,451.557121,452.494120,0.936998,0.207504,0.897443,...,False,0.978380,False,0.320896,False,0.577629,0.0,0.649638,0.0,0.625635
1,AASplitter┴rs 72┴,487.177694,487.227354,0.049660,0.010193,451.997959,452.074520,0.076561,0.016938,0.901919,...,False,0.763879,False,0.470399,False,0.914118,0.0,0.617139,0.0,0.716132
2,AASplitter┴rs 2┴,487.260717,487.142857,-0.117859,-0.024188,451.601964,452.479660,0.877696,0.194352,0.769932,...,False,0.922256,False,0.447445,False,0.529520,0.0,0.684851,0.0,0.633074
3,AASplitter┴rs 43┴,487.681346,486.738684,-0.942662,-0.193295,452.008833,452.062468,0.053635,0.011866,0.019354,...,False,0.142599,False,0.985370,False,0.483852,0.5,0.563985,0.0,0.537274


# AA Test with stratification

Depending on your requirements it is possible to stratify the data. You can set `stratification=True` and `StratificationRole` in `Dataset` to run it with stratification.

Stratified AA tests ensure that both groups (control/test) have the same proportions of categories (e.g. same % of genders or regions). This prevents imbalances in categorical features that can distort results.

Make sure to assign `StratificationRole` to relevant columns in your dataset before enabling stratification.

In [17]:
data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": StratificationRole(str),
    }, data=create_test_data(),
)
data

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry
0,0.0,4.0,1.0,489.0,514.444444,38.0,F,Logistics
1,1.0,0.0,0.0,493.5,406.0,52.0,M,Logistics
2,2.0,0.0,0.0,489.5,412.666667,66.0,F,E-commerce
3,3.0,0.0,0.0,490.0,411.444444,30.0,F,Logistics
4,4.0,2.0,1.0,518.0,511.222222,69.0,F,E-commerce
...,...,...,...,...,...,...,...,...
9995,9995.0,0.0,0.0,480.0,414.555556,31.0,M,Logistics
9996,9996.0,10.0,1.0,509.5,438.333333,43.0,F,E-commerce
9997,9997.0,0.0,0.0,474.5,419.888889,57.0,F,Logistics
9998,9998.0,4.0,1.0,484.5,510.333333,28.0,M,E-commerce


In [18]:
test = AATest(random_states=[56, 72, 2, 43], stratification=True)
result = test.execute(data)

100%|██████████| 4/4 [00:00<00:00,  6.83it/s]
[DEBUG _compute_weighted_pvalues] pval_cols = ['pre_spends TTest p-value test_1', 'post_spends TTest p-value test_1', 'pre_spends KSTest p-value test_1', 'post_spends KSTest p-value test_1', 'mean TTest p-value all', 'mean KSTest p-value all']
[DEBUG _compute_weighted_pvalues] col='pre_spends TTest p-value test_1', lookup_key='pre_spends TTest test_1', weight=0.95, NaN=0/4, values=[0.8509346553560393, 0.9206084383464133, 0.330517995255702, 0.4870532164860708]
[DEBUG _compute_weighted_pvalues] col='post_spends TTest p-value test_1', lookup_key='post_spends TTest test_1', weight=0.95, NaN=0/4, values=[0.7283218732804304, 0.9353428215299191, 0.5193549189647317, 0.8247164811586384]
[DEBUG _compute_weighted_pvalues] col='pre_spends KSTest p-value test_1', lookup_key='pre_spends KSTest test_1', weight=0.95, NaN=0/4, values=[0.807073423673699, 0.8557313879398134, 0.8847168133674044, 0.23677754373891566]
[DEBUG _compute_weighted_pvalues] col='post_

In [19]:
result.resume

,feature,group,TTest aa test,KSTest aa test,TTest best split,KSTest best split,result,control mean,test mean,difference,difference %
0,post_spends,test_1,OK,OK,OK,OK,OK,451.913496,451.845942,-0.067553,-0.014948
1,pre_spends,test_1,OK,OK,OK,OK,OK,487.496128,487.535929,0.039801,0.008164


In [20]:
result.aa_score

,score,pass
pre_spends TTest test_1,0.95,True
post_spends TTest test_1,0.95,True
pre_spends KSTest test_1,0.95,True
post_spends KSTest test_1,0.95,True


In [21]:
result.best_split

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry,split
0,0,4.0,1.0,489.0,514.444444,38.0,F,Logistics,control
1,1,0.0,0.0,493.5,406.0,52.0,M,Logistics,test_1
2,2,0.0,0.0,489.5,412.666667,66.0,F,E-commerce,test_1
3,3,0.0,0.0,490.0,411.444444,30.0,F,Logistics,test_1
4,4,2.0,1.0,518.0,511.222222,69.0,F,E-commerce,control
...,...,...,...,...,...,...,...,...,...
8996,9995,0.0,0.0,480.0,414.555556,31.0,M,Logistics,test_1
8997,9996,10.0,1.0,509.5,438.333333,43.0,F,E-commerce,test_1
8998,9997,0.0,0.0,474.5,419.888889,57.0,F,Logistics,control
8999,9998,4.0,1.0,484.5,510.333333,28.0,M,E-commerce,control


In [22]:
result.best_split_statistic

,feature,group,control mean,test mean,difference,difference %,TTest pass,TTest p-value,KSTest pass,KSTest p-value
0,post_spends,test_1,451.913496,451.845942,-0.067553,-0.014948,OK,0.935343,OK,0.979779
1,pre_spends,test_1,487.496128,487.535929,0.039801,0.008164,OK,0.920608,OK,0.855731


In [23]:
result.experiments

,splitter_id,pre_spends GroupDifference control mean test_1,pre_spends GroupDifference test mean test_1,pre_spends GroupDifference difference test_1,pre_spends GroupDifference difference % test_1,post_spends GroupDifference control mean test_1,post_spends GroupDifference test mean test_1,post_spends GroupDifference difference test_1,post_spends GroupDifference difference % test_1,pre_spends TTest p-value test_1,...,post_spends TTest pass test_1,pre_spends KSTest p-value test_1,pre_spends KSTest pass test_1,post_spends KSTest p-value test_1,post_spends KSTest pass test_1,mean TTest p-value,mean TTest pass,mean KSTest p-value,mean KSTest pass,mean test score
0,AASplitterWithStratification┴rs 56┴,487.554318,487.479244,-0.075075,-0.015398,451.732020,452.021251,0.289231,0.064027,0.850935,...,False,0.807073,False,0.770192,False,0.789628,0.0,0.788633,0.0,0.788965
1,AASplitterWithStratification┴rs 72┴,487.496128,487.535929,0.039801,0.008164,451.913496,451.845942,-0.067553,-0.014948,0.920608,...,False,0.855731,False,0.979779,False,0.927976,0.0,0.917755,0.0,0.921162
2,AASplitterWithStratification┴rs 2┴,487.323736,487.712424,0.388688,0.079760,451.614530,452.151102,0.536572,0.118812,0.330518,...,False,0.884717,False,0.531417,False,0.424936,0.0,0.708067,0.0,0.613690
3,AASplitterWithStratification┴rs 43┴,487.656956,487.379401,-0.277555,-0.056916,451.786134,451.970625,0.184491,0.040836,0.487053,...,False,0.236778,False,0.614328,False,0.655885,0.0,0.425553,0.0,0.502330


# AA Test by samples 

Depending on your requirements and size of data it is possible to estimate AA test on samples the data. You can set `sample_size=size` to run it. 

In [24]:
data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": StratificationRole(str),
    }, data=create_test_data(),
)
data

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry
0,0.0,0.0,0.0,477.0,428.222222,66.0,F,Logistics
1,1.0,2.0,1.0,511.0,525.333333,62.0,F,Logistics
2,2.0,9.0,1.0,485.5,456.444444,19.0,M,Logistics
3,3.0,6.0,1.0,467.5,492.444444,39.0,M,E-commerce
4,4.0,9.0,1.0,472.0,441.777778,49.0,M,E-commerce
...,...,...,...,...,...,...,...,...
9995,9995.0,0.0,0.0,481.0,421.555556,52.0,F,E-commerce
9996,9996.0,11.0,1.0,483.0,429.0,66.0,M,E-commerce
9997,9997.0,0.0,0.0,452.0,422.666667,66.0,F,E-commerce
9998,9998.0,9.0,1.0,490.5,443.888889,59.0,M,Logistics


In [25]:
test = AATest(n_iterations=10, sample_size=0.3)
result = test.execute(data)

100%|██████████| 10/10 [00:01<00:00,  7.52it/s]
[DEBUG _compute_weighted_pvalues] pval_cols = ['pre_spends TTest p-value nan', 'post_spends TTest p-value nan', 'pre_spends KSTest p-value nan', 'post_spends KSTest p-value nan', 'mean TTest p-value all', 'mean KSTest p-value all']
[DEBUG _compute_weighted_pvalues] col='pre_spends TTest p-value nan', lookup_key='pre_spends TTest nan', weight=0.95, NaN=10/10, values=[nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
[DEBUG _compute_weighted_pvalues] col='post_spends TTest p-value nan', lookup_key='post_spends TTest nan', weight=0.95, NaN=10/10, values=[nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
[DEBUG _compute_weighted_pvalues] col='pre_spends KSTest p-value nan', lookup_key='pre_spends KSTest nan', weight=0.95, NaN=10/10, values=[nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
[DEBUG _compute_weighted_pvalues] col='post_spends KSTest p-value nan', lookup_key='post_spends KSTest nan', weight=0.95, NaN=10/10, values=[nan, nan, nan,

KeyError: nan

In [ ]:
result.resume

In [ ]:
result.aa_score

In [ ]:
result.best_split

In [ ]:
result.best_split_statistic

In [ ]:
result.experiments

# AATest with Target Role for a categorical feature

It is possible to assign Target Role to categorical features. A categorical feature can also be the target or outcome variable. In this case, the Chi-square test is added to the pipeline of AATest.

In [ ]:
data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "treat": TreatmentRole(int),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": TargetRole(str)
    }, data=create_test_data(),
)
data

In [ ]:
test = AATest(n_iterations=10)
result = test.execute(data)

In [ ]:
result.resume

In [ ]:
result.aa_score

In [ ]:
result.best_split

In [ ]:
result.best_split_statistic

In [ ]:
result.experiments

# AATest with unequal group sizes

AATest can be performed to get a split with unequal the groups of different sizes by using `unequal_size` argument. Also Whelch correction can be applied by adding `t_test_equal_vat=False` argument while initiating AATest instance.

In [ ]:
test = AATest(n_iterations=10, control_size=0.3, t_test_equal_var=False)
result = test.execute(data)

In [ ]:
result.best_split.data.groupby("split").agg("count")

In [ ]:
result.best_split_statistic

# AAnTest

AAnTest is an extension of AATest that allows to split the dataset into several test groups, additionally to the control group.

In [ ]:
test = AATest(groups_sizes=[0.3, 0.2, 0.2, 0.3])
result = test.execute(data)

In [ ]:
result.best_split.data.groupby("split").agg("count")

In [ ]:
result.best_split_statistic

# AATest with partially pre-defined groups

Certain users can be pre-assigned to either the test or the control group, so that they are not randomly assigned. This can be done using the `ConstGroupRole` role. In order to pre-assign users to the control group they should have a value of `control`, and in the test group they should have a value of `test` in the column with the role `ConstGroupRole`. Users that are not pre-assigned to either the control or the test group should have `None`, so that they will be assigned randomly.

In [ ]:
pd_data= create_test_data()
pd_data.loc[pd_data["treat"]==0, "const_grp"] = "control"
pd_data.loc[pd_data["treat"]==1, "const_grp"] = "test"
pd_data.loc[2000:, "const_grp"] = None

data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "const_grp": ConstGroupRole(str),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": StratificationRole(str),
        "industry": TargetRole(str),
    }, data=pd_data,
)
data

In [ ]:
test = AATest(n_iterations=1)
result = test.execute(data)

In [ ]:
result.resume

In [ ]:
result.best_split

## Common issues and tips

- **Missing roles**: Make sure all target variables are assigned `TargetRole`. Columns without roles may cause silent failure.
- **Stratification**: If your dataset contains categorical features (e.g. `gender`, `region`) that may affect the outcome, use `StratificationRole` and enable `stratification=True` in `AATest(...)`.
- **Imbalanced categories**: If some categories have too few samples, stratified splits may become unstable. Consider filtering or merging rare categories.
- **Random fluctuations**: On small datasets, it's normal to see occasional `NOT OK` results. Use more iterations (e.g. `n_iterations=50`) for stability.
- **Missing values**: NaNs in stratification columns may be treated as separate categories. Clean or fill missing values before stratified AA tests.